# K5Lnegu: K5 + `L_neg` with one meaningless target per video, s2024, read beside K5 and K5Lneg

Rule: `core/docs/final/KIM_RUN.md` **§28.3**, written before any number. **PRELIMINARY (n = 1)**; K5 stays the main model. `nexar_test` is never touched. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

The question: K5Lneg (§28) cost 4 of 5 sets. Is the cost (A) the **one shared class text** every accident window is pulled to, or (B) the **pooled-feature objective** itself? K5Lnegu keeps (B) exactly (`ClassContrastLoss`, cosine / 0.07, mining, weight 1.0) and replaces (A): each source video gets **its own fixed random unit target** (other videos' accident windows become negatives). **Config diff to K5 = `loss.cap_contrast_mode=uniq` only.** Same corpus (`Nexar_window`), 1,749 steps, DVS off.

**U2:** A iff K5Lnegu − K5 is "below" on none of Nexar / DoTA-CAP-dev / DADA / DAD; B iff "below" on ≥ 3 and K5Lnegu − K5Lneg "above" on ≤ 1; else MIXED. TAD, P2 `sep` printed.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, preflight (K5, K5Lneg reads, bakes, census, Nexar_window) | no | 2-5 min |
| 4 | pytest on the touched code — **raises if red** | no | 2-3 min |
| 5 | check K5's corpus `Nexar_window` (read-only) | no | < 1 min |
| 6 | train K5Lnegu s2024 -> `outputs/kim_lneg_uniq/K5/s2024` + loss print | yes | about as long as K5 |
| 7 | zero-shot: DoTA-CAP-dev, DADA whole, TAD, DAD | yes | 20-30 min |
| 8 | `nexar_val` with `--save-scores` | yes | 10-20 min |
| 9 | P2 `feature_diversity` K5Lnegu vs K5 (printed) | yes | 10-20 min |
| 10 | `core.tools.k5_lneg_uniq_report` -> **paste `REPORTS/k5_lneg_uniq_report.md` to Claude** | no | 1 min |
| 11 | curves K5 / K5Lneg / K5Lnegu on `nexar_val` (illustration) | no | 3-5 min |

A re-run skips what is on Drive. Thesis and Thesis-V2 are only read; everything new goes to `Thesis-Final/outputs/kim_lneg_uniq/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes, Nexar caches, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5 and its reads, DAD; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '### 28.3 K5Lnegu' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §28.3 -- re-upload kat-vad/'
assert (REPO / 'core/tools/k5_lneg_uniq_report.py').is_file(), 'no k5_lneg_uniq_report -- re-upload kat-vad/'
assert 'def uniq_rows' in (REPO / 'core/losses/contrastive.py').read_text(), 'no uniq_rows -- re-upload kat-vad/'
assert (REPO / 'core/tools/feature_diversity.py').is_file(), 'no feature_diversity -- re-upload kat-vad/'
assert 'class ClassContrastLoss' in (REPO / 'core/losses/contrastive.py').read_text(), 'no ClassContrastLoss -- re-upload kat-vad/'
assert 'cap_contrast_mode' in (REPO / 'core/config.py').read_text(), 'config has no loss.cap_contrast_mode -- re-upload kat-vad/'
assert '--save-scores' in (REPO / 'core/tools/nexar_eval.py').read_text(), 'nexar_eval has no --save-scores -- re-upload kat-vad/'
assert 'VIDEO_STATS_JSON' in (REPO / 'core/tools/nexar_eval.py').read_text(), 'nexar_eval writes no video_stats -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5, NEG, LNEG = constants.KIM_MAIN_ARM, constants.KIM_NEG_ROW, constants.KIM_LNEG_ROW    # K5, K5neg, K5Lneg
LN       = constants.KIM_LNEG_UNIQ_ROW             # K5Lnegu: the new row
MODE     = constants.CAP_CONTRAST_MODE_UNIQ        # §28.3 U1: the one key
SEED     = constants.KIM_SEED                      # 2024
ENC      = constants.V5_ENCODER                    # V2-S
DATASET  = constants.NEXAR_DATASET
C_WINDOW = constants.NEXAR_CONSTRUCTION_WINDOW
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL         # nexar_test is never read here
TAD_NAME = constants.TAD_DATASET
DAD_NAME, DAD_STRIDE = constants.DAD_DATASET, constants.DAD_STRIDE

V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C_WINDOW}'     # K5's corpus: the reference of every assert
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                # nexar_train statistics, every train video
# K5: the row it is compared with
KIM      = constants.OUTPUT_ROOT / constants.KIM_OUTPUT_DIRNAME
K5_RUN   = KIM / K5 / f's{SEED}'
K5_REF   = constants.OUTPUT_ROOT / constants.KIM_T2_OUTPUT_DIRNAME / 'eval' / 'nexar' / K5   # §13's read (NG4)
# Zero-shot inputs: the bakes K5 was read on
A2_DADA  = V2_CACHE / 'v2' / 'A2_S' / f'{T2NAME}_test_sources_nexar'
A2_DOTA  = V2_CACHE / 'v2' / 'A2_S' / f'{CAP_DEV}_s3_from_s1_nexar'
A2_TAD   = V2_CACHE / 'v2' / 'A2_S' / f'{TAD_NAME}_s8_nexar'
A2_DAD   = constants.CACHE_ROOT / 'v2' / 'A2_S' / f'{DAD_NAME}_s{DAD_STRIDE}_nexar'
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
TAD      = constants.DATA_ROOT / TAD_NAME
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
# Nexar
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
NEXAR_U  = V2_CACHE / 'video' / ENC / f'{constants.NEXAR_VIDEO_S1_DATASET}_squash'
# K5neg (§17): its reads, §17's K5 re-read on nexar_val, and both rows' per-frame curves
NEG_ROOT = constants.OUTPUT_ROOT / constants.KIM_NEG_OUTPUT_DIRNAME
K5_SCORES  = constants.OUTPUT_ROOT / constants.KIM_GATE_OUTPUT_DIRNAME / 'eval' / 'nexar' / K5
NEG_SCORES = NEG_ROOT / 'eval' / constants.ZS_CURVES_NEXAR_DIRNAME / NEG
# Outputs
LNEG_ROOT = constants.OUTPUT_ROOT / constants.KIM_LNEG_OUTPUT_DIRNAME           # §28's K5Lneg: control 2
LNEG_CKPT = LNEG_ROOT / K5 / f's{SEED}' / 'checkpoint_last.pt'
ROOT     = constants.OUTPUT_ROOT / constants.KIM_LNEG_UNIQ_OUTPUT_DIRNAME
DATA     = WINDOW                                                # K5's own corpus, read-only (§28 NL1, kept by §28.3)
RUN      = ROOT / K5 / f's{SEED}'
CKPT     = RUN / 'checkpoint_last.pt'
EVAL     = ROOT / 'eval'
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/k5_lneg_uniq')
LOCAL.mkdir(parents=True, exist_ok=True)

INPUTS = (('Nexar_window labels', WINDOW / constants.LABELS_TRAIN_FILENAME), ('Nexar A2 bake', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('K5 checkpoint', K5_RUN / 'checkpoint_last.pt'), ('K5 verified', K5_RUN / 'verified.json'),
          ('K5 config', K5_RUN / 'config.yaml'), ('K5 nexar_val read (§13)', K5_REF / 'nexar_readout.json'),
          ('DADA A2 bake', A2_DADA), ('DoTA-CAP A2 bake', A2_DOTA), ('TAD A2 bake', A2_TAD), ('DAD A2 bake', A2_DAD),
          ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS), ('DADA annotation', ANNOT),
          ('DoTA s1', DOTA_S1), ('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME),
          ('Nexar census', CENSUS), ('Nexar CLIP s1', NEXAR_X), ('Nexar V2-S s1', NEXAR_U),
          ('K5 DoTA read', KIM / 'eval' / 'dota' / K5), ('K5 DADA read', KIM / 'eval' / 'dada' / K5),
          ('K5 TAD read', KIM / 'eval' / 'tad' / K5 / f's{SEED}' / 'results.json'),
          ('K5 DAD read', KIM / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json'),
          ('§17 K5 re-read', NEG_ROOT / 'eval' / 'nexar' / K5 / 'video_stats.json'),
          ('§17 K5neg read', NEG_ROOT / 'eval' / 'nexar' / NEG / 'video_stats.json'),
          ('K5neg DoTA read', NEG_ROOT / 'eval' / 'dota' / K5), ('K5neg DADA read', NEG_ROOT / 'eval' / 'dada' / K5),
          ('K5neg TAD read', NEG_ROOT / 'eval' / 'tad' / K5 / f's{SEED}' / 'results.json'),
          ('K5neg DAD read', NEG_ROOT / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json'),
          ('K5 nexar_val curves', K5_SCORES / 'scores'), ('K5neg nexar_val curves', NEG_SCORES / 'scores'),
          ('K5Lneg checkpoint', LNEG_CKPT), ('K5Lneg nexar_val read', LNEG_ROOT / 'eval' / 'nexar' / LNEG / 'nexar_readout.json'),
          ('K5Lneg DoTA read', LNEG_ROOT / 'eval' / 'dota' / K5), ('K5Lneg DADA read', LNEG_ROOT / 'eval' / 'dada' / K5),
          ('K5Lneg TAD read', LNEG_ROOT / 'eval' / 'tad' / K5 / f's{SEED}' / 'results.json'),
          ('K5Lneg DAD read', LNEG_ROOT / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json'))
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | corpus {DATA} (K5\'s) | runs/reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 6-8 need one'
df -h /content | tail -1

In [ ]:
import torch
import yaml

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(drive_dir, steps):
    rec, ckpt = drive_dir / 'verified.json', drive_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def manifest(d):
    return json.loads((d / constants.V2_INPUT_MANIFEST_FILENAME).read_text())



# Preflight on small files only (C39)
k5_cfg = _flat(yaml.safe_load((K5_RUN / 'config.yaml').read_text()))
assert k5_cfg['kim.enabled'] and k5_cfg['kim.prototypes'] and k5_cfg['kim.edge_mask'], 'K5_RUN is not K5'
assert k5_cfg['dvs.theta'] == 1.0 and k5_cfg['data.dataset'] == DATASET, 'K5_RUN is not K5 on Nexar'
STEPS = json.loads((K5_RUN / 'verified.json').read_text())['step']             # 1,749
EPOCHS = k5_cfg['train.num_epochs']                                            # 33
assert is_verified(K5_RUN, STEPS) and metrics_last(K5_RUN / 'metrics.jsonl')['global_step'] == STEPS, 'K5: J10'
SHA = manifest(A2_NEXAR)['train_ids_sha1']
for name, d in (('Nexar', A2_NEXAR), ('DADA', A2_DADA), ('DoTA-CAP', A2_DOTA), ('TAD', A2_TAD), ('DAD', A2_DAD)):
    m = manifest(d)
    assert (m['crn'], m['motion'], m['train_ids_sha1']) == (constants.V2_OFF, ENC, SHA), f'{name} bake: {m}'
census = json.loads(CENSUS.read_text())['videos']
TRAIN_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_TRAIN))
NEG_IDS = [v for v in TRAIN_IDS if census[v]['label'] == 0]
DADA_IDS = test_sources(T2)
CAP_IDS = sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
TAD_IDS = sorted(json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert (len(TRAIN_IDS), len(NEG_IDS)) == (900, 449), (len(TRAIN_IDS), len(NEG_IDS))
assert len(DADA_IDS) == 370 and len(TAD_IDS) == constants.V2_TADZ_VIDEOS and len(DAD_IDS) == constants.DAD_TEST_VIDEOS
for name, d, ids in (('Nexar train bake', A2_NEXAR, TRAIN_IDS), ('DADA bake', A2_DADA, DADA_IDS),
                     ('DoTA-CAP bake', A2_DOTA, CAP_IDS), ('TAD bake', A2_TAD, TAD_IDS), ('DAD bake', A2_DAD, DAD_IDS),
                     ('Nexar CLIP', NEXAR_X, NEXAR_IDS), ('Nexar V2-S', NEXAR_U, NEXAR_IDS)):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
print(f'preflight OK: K5 step {STEPS} ({EPOCHS} epochs) | train {len(TRAIN_IDS)} ({len(NEG_IDS)} negative) | '
      f'DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}, TAD {len(TAD_IDS)}, DAD {len(DAD_IDS)}, {NEXAR_SPLIT} {len(NEXAR_IDS)}')

## 4. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_class_contrast.py',
     'core/tests/test_train_units.py', 'core/tests/test_k5_lneg_uniq_report.py', 'core/tests/test_feature_diversity.py',
     'core/tests/test_grad_probe.py', 'core/tests/test_baseline_losses.py'])

## 5. K5's corpus `Nexar_window` (§28 NL1; nothing is built)

Asserts: the corpus holds K5's 7,311 train windows (1,689 positive, `KIM_RUN.md` §27 NC2), every positive window comes from a positive train video and no window from a negative one, and the step plan of a balanced epoch equals K5's.

In [ ]:
labels = json.loads((DATA / constants.LABELS_TRAIN_FILENAME).read_text())
windows = json.loads((DATA / constants.WINDOWS_FILENAME).read_text())
n_pos = sum(labels.values())
assert (len(labels), n_pos) == (7311, 1689), f'not K5\'s corpus: {len(labels)} windows, {n_pos} positive'
sources = {windows[w]['source'] for w in labels}
assert not sources & set(NEG_IDS), 'Nexar_window holds a negative-video window -- that is K5neg / K5nc2, not K5'
assert sources <= set(TRAIN_IDS), 'a window comes from outside nexar_train'
steps_per_epoch = -(-2 * n_pos // 64)            # balanced epoch = 2 x positives, batch 64
assert steps_per_epoch * EPOCHS == STEPS, f'{steps_per_epoch} x {EPOCHS} != K5 {STEPS}'
print(f'Nexar_window: {n_pos} positive + {len(labels) - n_pos} normal windows from {len(sources)} positive videos | '
      f'{steps_per_epoch} steps/epoch x {EPOCHS} = {STEPS}')

## 6. Train K5Lnegu, s2024 (§28.3 U1)

**Watch the log: past 2 s/step, stop and paste it.** Then J10, the config diff against K5 (**only** `loss.cap_contrast_mode` may differ) and the NL6 loss print: `cap_contrastive − cap_contrastive_floor` beside `mil` at step 1 and over the last epoch.

In [ ]:
K5_SETS = ['kim.enabled=true', 'kim.prototypes=true', 'kim.edge_mask=true', 'dvs.theta=1.0', 'dvs.theta_ego=1.0']
SETS = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
        f'data.dataset={DATASET}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
        'kip.enabled=false', 'loss.dvs_anchor_mode=span', 'data.is_egocentric=false',
        f'v2.motion={ENC}', f'model.motion_dim={constants.V5_GRID_DIM}',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}', *K5_SETS,
        f'loss.cap_contrast_mode={MODE}']                                  # §28.3 U1: the one change
DEFAULTS = _flat(Config().to_dict())


def config_diff(base, got):
    '''Keys that differ; a key added after ``base`` trained may sit at its default.'''
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


def check_run():
    payload = torch.load(CKPT, map_location='cpu', weights_only=False)
    assert payload['global_step'] == metrics_last(RUN / 'metrics.jsonl')['global_step'] == STEPS, 'J10'
    assert any(k.startswith('kim.') for k in payload['model']), 'no KIM weights in the checkpoint'
    got = _flat(yaml.safe_load((RUN / 'config.yaml').read_text()))
    diff = config_diff(k5_cfg, got)
    assert diff == {'loss.cap_contrast_mode': (None, MODE)}, f'K5Lnegu vs K5 must differ in one key: {diff}'
    assert got['loss.captions_from_definitions'] is False, 'the caption path must stay off'
    rows = [json.loads(l) for l in (RUN / 'metrics.jsonl').read_text().splitlines() if l.strip()]
    assert all('cap_contrastive' in r for r in rows), 'a logged step has no cap_contrastive: the term did not run'
    last_epoch = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    for name, sel in (('step 1', rows[:1]), ('last epoch', last_epoch)):
        m = {k: sum(r[k] for r in sel) / len(sel) for k in ('mil', 'cap_contrastive', 'cap_contrastive_floor', 'cap_mined')}
        print(f'NL6 {name}: mil {m["mil"]:.4f} | cap_contrastive {m["cap_contrastive"]:.4f} - floor '
              f'{m["cap_contrastive_floor"]:.4f} = {m["cap_contrastive"] - m["cap_contrastive_floor"]:+.4f} | mined {m["cap_mined"]:.1f}')
    print(f'K5Lnegu: J10 {STEPS} | config == K5 except loss.cap_contrast_mode={MODE}')


if is_verified(RUN, STEPS):
    print(f'K5Lnegu s{SEED}: verified on Drive -- not retrained')
else:
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'run'
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in SETS for x in ('--set', s)],
         '--data-dir', stage(DATA, LOCAL / 'stage' / 'data'), '--clip-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar'),
         '--output-dir', local, *resume], log=LOCAL / 'train.log')
    last = metrics_last(local / 'metrics.jsonl')
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'J10 locally: {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, RUN / name)
    flush_drive()
    assert ckpt_step(CKPT) == STEPS, f'Drive copy is not step {STEPS}'
    (LOCAL / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': CKPT.stat().st_size}))
    publish(LOCAL / 'verified.json', RUN / 'verified.json')
    print(f'trained in {(time.time() - t0) / 60:.0f} min | mil {last.get("mil")} | kim_mil {last.get("kim_mil")} '
          f'| W_out {last.get("kim_w_out_norm")} | proto cos {last.get("kim_proto_cos")}')
check_run()

## 7. K5Lnegu zero-shot: DoTA-CAP-dev (protocol B), DADA (whole), TAD, DAD — K5's protocols and bakes (§28 NL4)

In [ ]:
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_TAD = LOCAL / 'TAD_meta'
L_TAD.mkdir(parents=True, exist_ok=True)
for f in TAD.iterdir():                     # json / txt only -- never a frame folder
    if f.is_file() and f.suffix in ('.json', '.txt'):
        shutil.copy2(f, L_TAD / f.name)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)

out = EVAL / 'dota' / K5
if not (out / 'clip_aucs.json').exists():
    local = LOCAL / 'eval' / 'dota'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', CKPT,
         '--input-dir', stage(A2_DOTA, LOCAL / 'stage' / 'cap', CAP_IDS), '--s1-dir', stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS),
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels'), '--split', CAP_DEV,
         '--out-dir', local, '--device', 'auto'], log=LOCAL / 'dota.log')
    publish(local, out)
print(f"DoTA-CAP-dev: macro {json.loads((out / 'protocol_b_readout.json').read_text())['macro']['mean']:.4f}")

out = EVAL / 'dada' / K5
if not (out / 'clip_aucs.json').exists():
    local = LOCAL / 'eval' / 'dada'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', CKPT, '--t2-dir', L_T2,
         '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', stage(A2_DADA, LOCAL / 'stage' / 'dada', DADA_IDS),
         '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / 'dada.log')
    publish(local, out)
print(f"DADA whole: macro {json.loads((out / 'dada_readout.json').read_text())['macro']['mean']:.4f}")


def read_evaluate(name, dataset, data_dir, bake, ids, out):
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', CKPT, '--set', f'data.dataset={dataset}',
             '--data-dir', data_dir, '--clip-dir', stage(bake, LOCAL / 'stage' / f'{name}_bake', ids),
             '--score-norm', 'auto', '--save-scores', '--output-dir', local], log=LOCAL / f'{name}.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(ids), r
    print(f"{name}: macro {r['auc_macro']:.4f} over {r['auc_macro_videos']} | micro {r['auc']:.4f} | AP {r['ap']:.4f}")


read_evaluate('tad', TAD_NAME, L_TAD, A2_TAD, TAD_IDS, EVAL / 'tad' / K5 / f's{SEED}')
read_evaluate('dad', DAD_NAME, stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta'), A2_DAD, DAD_IDS,
              EVAL / constants.DAD_OUTPUT_SET / K5 / f's{SEED}')
flush_drive()

## 8. `nexar_val`: K5Lnegu with `--save-scores` (`--final` is never passed; K5's re-read is §17's)

In [ ]:
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON, VIDEO_STATS_JSON  # noqa: E402

out = EVAL / 'nexar' / LN
done = all((out / f).exists() for f in (READOUT_JSON, ITEM_AUCS_JSON, VIDEO_STATS_JSON)) and \
    len(list((out / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
if not done:
    L_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
    L_U = stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS)
    L_STATS = stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', [])     # manifest + statistics only
    local = LOCAL / 'eval' / 'nexar' / LN
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', CKPT, '--stats-dir', L_STATS,
         '--video-s1-dir', L_U, '--census', CENSUS, '--clip-s1-dir', L_X,
         '--data-dir', L_DEFS, '--split', NEXAR_SPLIT, '--save-scores', '--out-dir', local], log=LOCAL / 'nexar.log')
    publish(local, out)
    flush_drive()
r = json.loads((out / READOUT_JSON).read_text())
assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS), (r['split'], r['videos'])
assert r['global_steps'] == {f's{SEED}': STEPS} and Path(r['checkpoints'][f's{SEED}']) == CKPT, (r['global_steps'], r['checkpoints'])
print(f"{LN}: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | "
      f"whole macro {r['whole']['macro']['mean']:.4f} | clip AUC {r['whole']['clip_auc_max']:.4f}")

## 9. P2 feature diversity K5Lnegu vs K5 (printed, §28.3) and 10. the table -> **paste `REPORTS/k5_lneg_uniq_report.md` to Claude**

In [ ]:
FD = REPORTS / constants.FEATDIV_REPORT_STEM
FD_JSON = FD / f'{constants.FEATDIV_REPORT_STEM}.json'
if not FD_JSON.exists():
    local = LOCAL / 'featdiv'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.feature_diversity', '--base-name', K5, '--base-ckpt', K5_RUN / 'checkpoint_last.pt',
         '--new-name', LN, '--new-ckpt', CKPT, '--census', CENSUS,
         '--clip-s1-dir', stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS),
         '--video-s1-dir', stage(NEXAR_U, LOCAL / 'stage' / 'nexar_u', NEXAR_IDS),
         '--stats-dir', stage(A2_NEXAR, LOCAL / 'stage' / 'nexar_stats', []), '--data-dir', L_DEFS,
         '--seed', SEED, '--out-dir', local], log=LOCAL / 'featdiv.log')
    publish(local, FD)
    flush_drive()
print((FD / f'{constants.FEATDIV_REPORT_STEM}.md').read_text())

local = LOCAL / 'report'
run([sys.executable, '-m', 'core.tools.k5_lneg_uniq_report', '--kim-root', KIM, '--lneg-root', LNEG_ROOT, '--uniq-root', ROOT,
     '--k5-nexar-dir', NEG_ROOT / 'eval' / 'nexar' / K5, '--k5-reference-dir', K5_REF, '--featdiv-json', FD_JSON,
     '--seed', SEED, '--out-dir', local])
publish(local, REPORTS)
print((REPORTS / f'{constants.KIM_LNEG_UNIQ_REPORT_STEM}.md').read_text())
flush_drive()

## 11. Curves on `nexar_val`: K5 / K5Lneg / K5Lnegu (illustration, in-domain, decides nothing)

Rank 1 = K5Lnegu's worst per-video AUC; all-normal videos last.

In [ ]:
from IPython.display import Image, display  # noqa: E402

TAG = 'k5_vs_k5lneg_vs_k5lnegu_s2024'
FIGS = KIM / 'REPORTS' / constants.ZS_CURVES_FIGURES_DIRNAME / 'train_nexar' / 'nexar' / TAG
LABEL = {LN: f'{constants.ZS_CURVES_MAIN_LABEL} L_neg uniq ({LN})', K5: f'{constants.ZS_CURVES_MAIN_LABEL} ({K5})',
         LNEG: f'{constants.ZS_CURVES_MAIN_LABEL} L_neg ({LNEG})'}
local = LOCAL / 'figures'
shutil.rmtree(local, ignore_errors=True)
reads = ['--read', 'nexar', LABEL[LN], EVAL / 'nexar' / LN, '--read', 'nexar', LABEL[K5], K5_SCORES,
         '--read', 'nexar', LABEL[LNEG], LNEG_ROOT / 'eval' / 'nexar' / LNEG]
common = ['--rank-by', LABEL[LN], '--name', TAG, '--out-dir', local, '--seed-label', f's{SEED}, n = 1',
          '--domain', constants.ZS_CURVES_DOMAIN_IN]
run([sys.executable, '-m', 'core.tools.zs_curves', *common, *reads])            # grid_nexar.png + onset_aligned.png
run([sys.executable, '-m', 'core.tools.zs_curves', '--all', *common, *reads])   # videos/nexar/<rank>_<id>.png
publish(local, FIGS)
for name in ('onset_aligned.png', 'grid_nexar.png'):
    display(Image(filename=str(local / name)))
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- figures in {FIGS}; paste REPORTS/k5_lneg_uniq_report.md to Claude')